# 03. 효율화와 분산 학습 점검 — 측정으로 확인하는 것, 계산으로 보는 것, 클러스터에서만 가능한 것

**등급**: 🟢 Mac 로컬 (MPS + CPU 2프로세스) · Colab T4에서도 실행 확인(약 2분; ESM-2 150M·650M `config.json`이 캐시에 있어야 함) · 🔴 SLURM 템플릿은 **클러스터에서 실행하지 못했다** · **예상 시간**: 약 1~2분 · **다운로드**: 없음

**질문**
1. 어텐션 구현(`sdpa` vs `eager`)·정밀도(fp32 vs bf16)·그래디언트 체크포인팅은 이 Mac(MPS)에서 시간과 메모리를 실제로 얼마나 바꾸는가? 서열 길이가 길어지면?
2. 전체 파인튜닝과 LoRA의 **메모리 예산**을 모델 크기별로 계산하면? (측정이 아니라 산술)
3. 다중 프로세스 학습이 **단일 프로세스와 같은 업데이트**를 만드는가? 데이터를 쪼개지 않으면 어떻게 틀어지는가? (`templates/train_ddp.py`를 CPU 2프로세스로 실행)

**설계 (결과를 보기 전에 정한 것)**
- §1: ESM-2 35M, 배치 4, 길이 {128, 256, 512, 1024}, 설정 4종 {sdpa fp32, eager fp32, sdpa bf16-autocast, sdpa fp32 + 체크포인팅}. 지표 = forward+backward 중앙값 시간(3회), **MPS 드라이버 할당 메모리**(PyTorch MPS는 최대 할당량 API가 없어 `driver_allocated_memory`를 대리 지표로 쓴다: 캐시된 버퍼를 포함하므로 그 설정의 최대 작업 집합에 가깝지만 peak와 같지는 않고, 각 설정 전에 `empty_cache`로 비운다). 각 설정 전에 `empty_cache`. 수치 동등성: sdpa와 eager의 마지막 층 출력 최대 절대차.
- §2: 모델 구성(`config.json`)에서 매개변수를 센 뒤 바이트로 환산하는 공식. **활성값(activation)은 제외**하고, 측정값이 아니다.
- §3: `torchrun --nproc_per_node=2`(gloo, CPU)로 4스텝(rank당 배치 8) vs `--nproc_per_node=1`(배치 16). 같은 합성 데이터·시드·SGD. 기대: 두 실행의 학습 매개변수가 거의 같다(부동소수점 합 순서 차이만). 음성 대조: `--no-sampler`(두 rank가 같은 샘플을 봄).

> **macOS 주의**: `torchrun --standalone`만 쓰면 이 Mac에서는 로컬 호스트 이름이 `1.0.0.127.in-addr.arpa`로 풀려 rendezvous가 300초 뒤 시간 초과로 실패했다(`DistNetworkError`). `--local-addr=127.0.0.1`을 주면 해결된다(아래 `run_ddp`).
>
> MPS에는 FlashAttention이 없다(CUDA 커널). 여기서 `sdpa`는 PyTorch의 `scaled_dot_product_attention`이 MPS에서 고르는 구현이고, 어떤 커널인지는 확인하지 못했다. 아래 수치를 CUDA GPU에 외삽하지 않는다.

In [1]:
import gc
import importlib.metadata as md
import json
import os
import subprocess
import sys
import tempfile
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "SETUP.md").exists():
            return q
    return None


ROOT = find_root()
CACHE = (ROOT / ".cache") if ROOT is not None else Path("cache")
os.environ.setdefault("HF_HOME", str(CACHE / "hf"))
TEMPLATES = (ROOT / "06_advanced_research" / "templates") if ROOT is not None else Path("templates")

import torch
from huggingface_hub import snapshot_download
from transformers import AutoConfig, AutoTokenizer, EsmModel
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("device:", DEVICE, "| templates:", TEMPLATES)
print({k: md.version(k) for k in ["numpy", "pandas", "torch", "transformers", "peft"]})

W1110 13:19:29.276000 11359 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


device: mps | templates: <repo>/06_advanced_research/templates
{'numpy': '2.5.4', 'pandas': '3.1.1', 'torch': '2.15.0', 'transformers': '5.22.1', 'peft': '0.21.2'}


## 1. 어텐션 구현·정밀도·체크포인팅 (측정)
입력은 무작위 아미노산 서열(내용은 속도와 무관). 손실은 마지막 층 출력의 제곱 평균(모든 매개변수에 기울기가 흐르게 함). 옵티마이저 스텝은 빼고 forward+backward만 잰다.

In [2]:
ESM35 = ("facebook/esm2_t12_35M_UR50D", "6fbf070e65b0b7291e7bbcd451118c216cff79d8")
D35 = Path(snapshot_download(ESM35[0], revision=ESM35[1], allow_patterns=["model.safetensors", "*.json", "vocab.txt"]))
TOK = AutoTokenizer.from_pretrained(D35)
AA = np.array(list("ACDEFGHIKLMNPQRSTVWY"))


def rand_batch(bs, L, seed=0):
    rng = np.random.default_rng(seed)
    seqs = ["".join(rng.choice(AA, size=L - 2)) for _ in range(bs)]                # <cls>와 <eos>를 합쳐 길이 L
    return TOK(seqs, return_tensors="pt")["input_ids"].to(DEVICE)


def sync():
    if DEVICE.type == "mps":
        torch.mps.synchronize()
    elif DEVICE.type == "cuda":
        torch.cuda.synchronize()


def bench(impl, precision, ckpt, L, bs=4, reps=3):
    gc.collect()
    if DEVICE.type == "mps":
        torch.mps.empty_cache()
    elif DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()                                           # (이 Mac에서는 실행되지 않는 경로: 미검증)
    model = EsmModel.from_pretrained(D35, add_pooling_layer=False, attn_implementation=impl).to(DEVICE).train()
    if ckpt:
        model.gradient_checkpointing_enable()
    ids = rand_batch(bs, L)
    times = []
    if DEVICE.type == "mps":
        torch.mps.empty_cache()
    for r in range(reps + 1):
        t0 = time.time()
        if precision == "bf16":
            with torch.autocast(DEVICE.type, dtype=torch.bfloat16):
                out = model(input_ids=ids, attention_mask=torch.ones_like(ids)).last_hidden_state
        else:
            out = model(input_ids=ids, attention_mask=torch.ones_like(ids)).last_hidden_state
        out.float().pow(2).mean().backward()
        sync()
        times.append(time.time() - t0)
        model.zero_grad(set_to_none=True)
    mem = torch.mps.driver_allocated_memory() / 1e9 if DEVICE.type == "mps" else (torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == "cuda" else np.nan)
    del model, out
    gc.collect()
    if DEVICE.type == "mps":
        torch.mps.empty_cache()
    return float(np.median(times[1:])), mem


# 수치 동등성: sdpa vs eager (fp32, L=256)
ids = rand_batch(4, 256)
outs = {}
for impl in ["sdpa", "eager"]:
    m = EsmModel.from_pretrained(D35, add_pooling_layer=False, attn_implementation=impl).to(DEVICE).eval()
    with torch.no_grad():
        outs[impl] = m(input_ids=ids, attention_mask=torch.ones_like(ids)).last_hidden_state.float().cpu()
    del m
print("sdpa vs eager 마지막 층 출력 최대 절대차(fp32, L=256): %.2e (출력 크기 중앙값 %.2f)" % ((outs["sdpa"] - outs["eager"]).abs().max().item(), outs["sdpa"].abs().median().item()))
del outs, ids                                                                           # 점검에 쓴 버퍼가 첫 측정에 섞이지 않게 비운다
gc.collect()
if DEVICE.type == "mps":
    torch.mps.empty_cache()

CONFIGS = {"sdpa fp32": ("sdpa", "fp32", False), "eager fp32": ("eager", "fp32", False), "sdpa bf16-autocast": ("sdpa", "bf16", False), "sdpa fp32 + 체크포인팅": ("sdpa", "fp32", True)}
LENS = [128, 256, 512, 1024]
rows = []
t0 = time.time()
for L in LENS:
    for name, (impl, prec, ck) in CONFIGS.items():
        try:
            sec, mem = bench(impl, prec, ck, L)
        except Exception as e:                                                      # 메모리 부족 등은 기록하고 계속한다
            sec, mem = np.nan, np.nan
            print("  실패:", name, L, str(e)[:80])
        rows.append(dict(길이=L, 설정=name, 초=sec, MPS_드라이버_GB=mem))
    print(f"길이 {L} 완료 (누적 {time.time() - t0:.0f}초)", flush=True)
BENCH = pd.DataFrame(rows)
print(BENCH.pivot(index="길이", columns="설정", values="초").round(2).to_string())
print()
print(BENCH.pivot(index="길이", columns="설정", values="MPS_드라이버_GB").round(2).to_string())

sdpa vs eager 마지막 층 출력 최대 절대차(fp32, L=256): 6.20e-06 (출력 크기 중앙값 0.12)


길이 128 완료 (누적 3초)


길이 256 완료 (누적 8초)


길이 512 완료 (누적 17초)


길이 1024 완료 (누적 38초)


설정    eager fp32  sdpa bf16-autocast  sdpa fp32  sdpa fp32 + 체크포인팅
길이                                                                
128         0.12                0.12       0.12               0.14
256         0.17                0.26       0.22               0.23
512         0.42                0.47       0.47               0.53
1024        1.04                1.17       1.03               1.38

설정    eager fp32  sdpa bf16-autocast  sdpa fp32  sdpa fp32 + 체크포인팅
길이                                                                
128         0.63                0.68       0.63               0.41
256         1.31                1.40       1.33               1.31
512         3.50                2.45       3.50               1.35
1024        7.84                7.90       7.85               2.48


## 2. 메모리 예산 (계산)
전체 파인튜닝(AdamW, fp32): 가중치 4P + 기울기 4P + 옵티마이저 상태 8P = **16P 바이트**. LoRA(q,v, r=8): 동결 가중치 4P + 학습 매개변수 $P_{\rm LoRA}=4\,n_{\rm layer}\,d\,r$에 대해 16$P_{\rm LoRA}$. 동결 가중치를 bf16으로 두면 $2P+16P_{\rm LoRA}$(*동결* 가중치만 bf16이고 LoRA 매개변수는 fp32로 학습하는 구성; 전체 파인튜닝을 혼합 정밀도로 하면 fp32 마스터 사본이 따로 필요해 총 바이트가 순수 fp32와 비슷한 16P 안팎†). **활성값·CUDA 컨텍스트·단편화는 제외**한 하한이며, 측정이 아니라 공식이다. 매개변수 수는 로컬 캐시 모델의 `config.json`으로 센다(메타 장치에서 구성만 하고 가중치는 읽지 않는다).

In [3]:
REPOS = {"8M": "facebook/esm2_t6_8M_UR50D", "35M": "facebook/esm2_t12_35M_UR50D", "150M": "facebook/esm2_t30_150M_UR50D", "650M": "facebook/esm2_t33_650M_UR50D"}
rows = []
for name, repo in REPOS.items():
    cfg = AutoConfig.from_pretrained(repo, local_files_only=True)
    with torch.device("meta"):
        P = sum(p.numel() for p in EsmModel(cfg, add_pooling_layer=False).parameters())
    r = 8
    PL = 4 * cfg.num_hidden_layers * cfg.hidden_size * r
    rows.append(dict(모델=name, 매개변수_M=P / 1e6, 층=cfg.num_hidden_layers, 은닉=cfg.hidden_size, LoRA_학습_M=PL / 1e6, 전체FT_GB=16 * P / 1e9, LoRA_fp32_GB=(4 * P + 16 * PL) / 1e9, LoRA_bf16동결_GB=(2 * P + 16 * PL) / 1e9))
BUD = pd.DataFrame(rows)
print(BUD.round(3).to_string(index=False))
print("\n(참고: 이름 그대로 3B·15B 모델은 로컬에 없어 세지 않았다. 같은 공식에 매개변수 수만 넣으면 된다 — 과제 3.)")

  모델  매개변수_M  층   은닉  LoRA_학습_M  전체FT_GB  LoRA_fp32_GB  LoRA_bf16동결_GB
  8M   7.409  6  320      0.061    0.119         0.031           0.016
 35M  33.270 12  480      0.184    0.532         0.136           0.069
150M 147.729 30  640      0.614    2.364         0.601           0.305
650M 649.401 33 1280      1.352   10.390         2.619           1.320

(참고: 이름 그대로 3B·15B 모델은 로컬에 없어 세지 않았다. 같은 공식에 매개변수 수만 넣으면 된다 — 과제 3.)


## 3. 분산 학습이 정말 같은 업데이트를 만드는가 (CPU 2프로세스)
`templates/train_ddp.py`: ESM-2 8M + LoRA, 합성 데이터, SGD, `DistributedSampler`(shuffle 끔), 평균 손실.
- **DDP 2프로세스 × rank당 8** vs **단일 프로세스 × 16**: rank r가 샘플 r, r+2, …를 받으므로 한 스텝에서 두 rank의 합집합이 단일 프로세스의 배치와 같다. 따라서 평균된 기울기가 같아야 한다.
- **음성 대조**: `--no-sampler`로 두 rank가 같은 샘플 8개를 보게 한다. 이 경우 단일 프로세스(16개)와 달라야 한다.
- 점검 항목: 각 실행의 학습 매개변수 최대 절대차, 그리고 **모든 rank의 매개변수 해시가 같은지**.

In [4]:
def run_ddp(nproc, per_rank, extra=(), tag="run", steps=4):
    tmp = Path(tempfile.mkdtemp())
    out, pnp = tmp / f"{tag}.json", tmp / f"{tag}.npy"
    cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone", "--local-addr=127.0.0.1", f"--nproc_per_node={nproc}", str(TEMPLATES / "train_ddp.py"),
           "--steps", str(steps), "--per-rank-batch", str(per_rank), "--out", str(out), "--save-params", str(pnp), *extra]
    # GPU가 있어도 CPU 2프로세스(gloo)로 확인한다: GPU 1개에서 rank 1이 cuda:1을 열지 못해 실패한다 (Colab T4에서 겪었다)
    env = dict(os.environ, OMP_NUM_THREADS="2", TOKENIZERS_PARALLELISM="false", CUDA_VISIBLE_DEVICES="")
    t0 = time.time()
    r = subprocess.run(cmd, capture_output=True, text=True, env=env, timeout=300)
    if r.returncode != 0:
        raise RuntimeError(r.stderr[-1500:])
    return json.loads(out.read_text()), np.load(pnp), time.time() - t0


res_init, p_init, _ = run_ddp(1, 16, tag="init", steps=0)             # 0스텝: 초기 매개변수
res_init2, p_init2, _ = run_ddp(2, 8, tag="init2", steps=0)           # 2프로세스의 초기 매개변수
res_single, p_single, t1 = run_ddp(1, 16, tag="single")
res_ddp, p_ddp, t2 = run_ddp(2, 8, tag="ddp")
res_bad, p_bad, t3 = run_ddp(2, 8, extra=("--no-sampler",), tag="nosampler")
print(f"실행 시간(CPU, 모델 로딩 포함): 단일 {t1:.0f}초 | DDP 2프로세스 {t2:.0f}초 | 샘플러 없음 {t3:.0f}초")
print("학습 매개변수 개수:", p_single.size)
rows = [
    dict(실행="단일 프로세스 (배치 16)", 손실=" ".join(f"{x:.4f}" for x in res_single["losses"]), rank_해시_모두_같음=res_single["all_ranks_equal"], 단일_대비_최대절대차=0.0),
    dict(실행="DDP 2프로세스 (rank당 8, 샘플러 사용)", 손실=" ".join(f"{x:.4f}" for x in res_ddp["losses"]), rank_해시_모두_같음=res_ddp["all_ranks_equal"], 단일_대비_최대절대차=float(np.abs(p_ddp - p_single).max())),
    dict(실행="[음성 대조] DDP 2프로세스, 샘플러 없음", 손실=" ".join(f"{x:.4f}" for x in res_bad["losses"]), rank_해시_모두_같음=res_bad["all_ranks_equal"], 단일_대비_최대절대차=float(np.abs(p_bad - p_single).max())),
]
print(pd.DataFrame(rows).to_string(index=False))
print("\n단일 실행에서 4스텝 동안 학습 매개변수가 움직인 최대 절대량(초기값 대비): %.2e → 위 표의 차이를 이 값과 비교해 읽는다" % np.abs(p_single - p_init).max())
print("초기 매개변수: 단일 vs 2프로세스 최대 절대차 %.1e | 2프로세스에서 rank 해시 일치: %s" % (np.abs(p_init2 - p_init).max(), res_init2["all_ranks_equal"]))
print("rank별 해시(DDP):", res_ddp["param_hash_per_rank"])
print("※ 손실 첫 값은 rank 0의 배치 평균이다. DDP의 rank 0은 샘플 0,2,4,… 8개, 단일은 0..15 16개라 첫 손실 값이 다른 것이 정상이다.")

실행 시간(CPU, 모델 로딩 포함): 단일 5초 | DDP 2프로세스 5초 | 샘플러 없음 5초
학습 매개변수 개수: 61761
                         실행                          손실  rank_해시_모두_같음  단일_대비_최대절대차
            단일 프로세스 (배치 16) 1.5608 0.8061 1.2701 0.9814           True 0.000000e+00
DDP 2프로세스 (rank당 8, 샘플러 사용) 1.0070 1.2782 1.2209 0.8342           True 3.725290e-09
  [음성 대조] DDP 2프로세스, 샘플러 없음 1.3223 1.7398 1.0848 0.5211           True 4.672986e-04

단일 실행에서 4스텝 동안 학습 매개변수가 움직인 최대 절대량(초기값 대비): 1.32e-02 → 위 표의 차이를 이 값과 비교해 읽는다
초기 매개변수: 단일 vs 2프로세스 최대 절대차 0.0e+00 | 2프로세스에서 rank 해시 일치: True
rank별 해시(DDP): ['4d2c2ea2f71ad9e4', '4d2c2ea2f71ad9e4']
※ 손실 첫 값은 rank 0의 배치 평균이다. DDP의 rank 0은 샘플 0,2,4,… 8개, 단일은 0..15 16개라 첫 손실 값이 다른 것이 정상이다.


## 정리
(실행 결과를 확인한 뒤 README §5에서 수치와 함께 해석한다. 이 노트북은 Mac MPS 한 대의 측정, 공식에 의한 메모리 하한, CPU gloo 2프로세스의 동작 점검이다. **GPU 클러스터·NCCL·다중 노드·FSDP/ZeRO는 실행하지 못했다.** `slurm_ddp.sbatch`는 `bash -n` 문법 점검만 했다.)